# 🎓 Chatbot Tư Vấn Tuyển Sinh FPT School

**RAG Chatbot** sử dụng:
- 🤖 LLM: `Qwen/Qwen2.5-1.5B-Instruct`
- 🔍 Embedding: `intfloat/multilingual-e5-small`
- 🗄️ Vector DB: FAISS
- 🖥️ UI: Gradio

---
> ⚡ **Khuyến nghị**: Chạy với **GPU T4** (Runtime → Change runtime type → T4 GPU)

## 🔧 Bước 1: Kiểm tra GPU & môi trường

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Không có GPU. Chạy bằng CPU sẽ chậm hơn (~3–5 phút/câu)')
    print('   → Vào Runtime > Change runtime type > chọn T4 GPU')

## 📦 Bước 2: Clone repository từ GitHub

In [ ]:
import os

REPO_URL = 'https://github.com/Luanntc2/chatbot-tuyen-sinh.git'
PROJECT_DIR = '/content/chatbot-tuyen-sinh'

if os.path.exists(PROJECT_DIR):
    print('🔄 Repo đã tồn tại, pulling updates...')
    os.system(f'git -C {PROJECT_DIR} pull')
else:
    print('📥 Đang clone repository...')
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print(f'\n✅ Working directory: {os.getcwd()}')
os.system('ls -la')

## 🐍 Bước 3: Cài đặt dependencies

> ⏳ Bước này mất khoảng **3–5 phút**

In [ ]:
import os, sys, torch

os.chdir('/content/chatbot-tuyen-sinh')

# Chọn faiss-gpu hoặc faiss-cpu tùy môi trường
faiss_pkg = 'faiss-gpu' if torch.cuda.is_available() else 'faiss-cpu>=1.7.4'
print(f'📦 Sẽ cài: {faiss_pkg}')

# Đọc requirements, loại dòng faiss
with open('requirements.txt') as f:
    lines = [l for l in f if not l.strip().startswith('faiss') and not l.strip().startswith('#')]
reqs = ''.join(lines)

with open('/tmp/req_colab.txt', 'w') as f:
    f.write(reqs)

os.system(f'pip install -q {faiss_pkg}')
os.system('pip install -q -r /tmp/req_colab.txt')
print('\n✅ Cài xong tất cả dependencies!')

## 🗄️ Bước 4: Build FAISS Vector Index

Nếu index đã có trong repo, bước này bỏ qua tự động.

In [ ]:
import os, sys
sys.path.insert(0, '/content/chatbot-tuyen-sinh')
os.chdir('/content/chatbot-tuyen-sinh')

from config import FAISS_INDEX_PATH

if os.path.exists(os.path.join(FAISS_INDEX_PATH, 'index.faiss')):
    print(f'✅ FAISS index đã tồn tại: {FAISS_INDEX_PATH}')
    print('   Bỏ qua bước build.')
else:
    print('🔄 Đang build FAISS index từ dữ liệu...')
    from src.ingest import ingest
    ingest()
    print('✅ Build FAISS index hoàn tất!')

## 🤖 Bước 5: Load RAG Pipeline

In [ ]:
import os, sys
sys.path.insert(0, '/content/chatbot-tuyen-sinh')
os.chdir('/content/chatbot-tuyen-sinh')

print('🔄 Đang load pipeline (lần đầu mất 2–3 phút để download model)...')
from src.pipeline import RAGPipeline

pipeline = RAGPipeline()
print('\n✅ Pipeline sẵn sàng!')

## 🧪 Bước 6 (Tùy chọn): Test nhanh CLI

In [ ]:
test_question = 'Học phí FPT School là bao nhiêu?'

print(f'❓ Câu hỏi: {test_question}')
print('─' * 60)
print('💬 Câu trả lời:')

for kind, value in pipeline.stream(test_question):
    if kind == 'token':
        print(value, end='', flush=True)
    elif kind == 'answer':
        print(value, end='', flush=True)

print('\n' + '─' * 60)

## 🚀 Bước 7: Chạy Gradio Web UI

Gradio sẽ tạo **public link** (dạng `https://xxxx.gradio.live`) để truy cập từ bất kỳ đâu.

> ⏱️ Link có hiệu lực trong **72 giờ**

In [ ]:
import os, sys
sys.path.insert(0, '/content/chatbot-tuyen-sinh')
os.chdir('/content/chatbot-tuyen-sinh')

import app as chatbot_app

# Inject pipeline đã load (tránh load lại)
chatbot_app.pipeline = pipeline

print('🚀 Khởi động Gradio...')
print('📌 Click vào link "Running on public URL" bên dưới để mở chatbot')
print('─' * 60)

chatbot_app.demo.launch(
    share=True,
    debug=False,
    show_error=True
)

---

## 🛠️ Xử lý sự cố

### ❌ CUDA out of memory
```python
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Chạy dòng trên rồi Restart & Run All
```

### ❌ ModuleNotFoundError
```bash
!pip install -q <tên_module>
# Sau đó Runtime → Restart session → chạy lại từ Bước 3
```

### ❌ Gradio không tạo được public URL
```python
# Thay share=True bằng share=False để xem trong Colab output
chatbot_app.demo.launch(share=False)
```

### 🔁 Reset hoàn toàn
Runtime → Disconnect and delete runtime → chạy lại từ đầu.

## 🔄 (Tùy chọn) Rebuild FAISS index khi có dữ liệu mới

In [ ]:
# ⚠️ Chỉ chạy khi đã thêm file mới vào data/raw/
import shutil, os, sys
sys.path.insert(0, '/content/chatbot-tuyen-sinh')
os.chdir('/content/chatbot-tuyen-sinh')
from config import FAISS_INDEX_PATH

if os.path.exists(FAISS_INDEX_PATH):
    shutil.rmtree(FAISS_INDEX_PATH)
    print(f'🗑️ Đã xóa index cũ')

from src.ingest import ingest
ingest()
print('✅ Rebuild hoàn tất!')